In [4]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from datetime import datetime

# ==============================
# 기본 설정
# ==============================
headers = {
    "User-Agent": "Mozilla/5.0"
}


# ==============================
# 1. 날짜 변환 함수
# ==============================
def format_date(date_str):
    """
    '2026.03.26. 오후 4:12' → '2026-03-26 16:12:00'
    변환 실패 시 원본 반환
    """
    try:
        date_str = date_str.replace("오전", "AM").replace("오후", "PM")
        dt = datetime.strptime(date_str, "%Y.%m.%d. %p %I:%M")
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except:
        return date_str


# ==============================
# 2. 기사 리스트 (title + url)
# ==============================
def get_article_links(page):
    url = f"https://news.naver.com/main/list.naver?mode=LSD&mid=sec&sid1=101&page={page}"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    articles = []
    news_list = soup.select("ul.type06_headline li, ul.type06 li")

    for item in news_list:
        dt_tags = item.select("dt")

        if len(dt_tags) == 0:
            continue

        a_tag = dt_tags[-1].select_one("a")

        if a_tag:
            title = a_tag.get_text(strip=True)
            link = a_tag["href"]

            if link.startswith("https://n.news.naver.com"):
                articles.append((title, link))

    return articles


# ==============================
# 3. 기사 상세 (content, date, press)
# ==============================
def get_article_detail(url):
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, "html.parser")

    # 본문
    content = ""
    content_tag = soup.select_one("#dic_area")

    if content_tag:
        # 🔥 불필요 요소 제거 (광고, 사진출처 등)
        for tag in content_tag.select("span.end_photo_org, div.ad, script, style"):
            tag.decompose()

        # 🔥 줄바꿈 유지하면서 텍스트 추출
        content = content_tag.get_text("\n", strip=True)

    # 날짜
    date_tag = soup.select_one("span.media_end_head_info_datestamp_time")
    date = date_tag.get_text(strip=True) if date_tag else ""
    date = format_date(date)

    # 언론사 (fallback 구조)
    press = ""

    press_tag = soup.select_one("a.media_end_head_top_logo")
    if press_tag:
        press = press_tag.get_text(strip=True)

    if not press:
        press_tag = soup.select_one("img.media_end_head_top_logo_img")
        if press_tag:
            press = press_tag.get("alt", "")

    if not press:
        press_tag = soup.select_one("span.writing")
        if press_tag:
            press = press_tag.get_text(strip=True)

    return content, date, press


# ==============================
# 4. 크롤링 실행
# ==============================
def crawl_naver_news(target_count=1000):
    all_data = []
    seen_links = set()
    page = 1

    while len(all_data) < target_count:
        print(f"📄 {page} 페이지 크롤링 중... ({len(all_data)}개 수집됨)")

        articles = get_article_links(page)

        if not articles:
            break

        for title, link in articles:

            if link in seen_links:
                continue
            seen_links.add(link)

            content, date, press = get_article_detail(link)

            # 빈 데이터 필터링 (품질 향상)
            if not title or not content:
                continue

            all_data.append({
                "title": title,
                "content": content,
                "date": date,
                "press": press,
                "url": link
            })

            if len(all_data) >= target_count:
                break

            time.sleep(0.5)

        page += 1

    return all_data


# ==============================
# 5. CSV 저장
# ==============================
def save_to_csv(data):
    filename = "news_dataset_economy.csv"

    print(f"\n📊 총 데이터 수: {len(data)}개")
    save = input("저장하시겠습니까? (y/n): ")

    if save.lower() != 'y':
        print("❌ 저장 취소")
        return

    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["title", "content", "date", "press", "url"])
        writer.writeheader()
        writer.writerows(data)

    print(f"✅ 저장 완료: {filename}")


# ==============================
# 실행
# ==============================
if __name__ == "__main__":
    data = crawl_naver_news(target_count=1000)
    save_to_csv(data)

📄 1 페이지 크롤링 중... (0개 수집됨)
📄 2 페이지 크롤링 중... (19개 수집됨)
📄 3 페이지 크롤링 중... (38개 수집됨)
📄 4 페이지 크롤링 중... (58개 수집됨)
📄 5 페이지 크롤링 중... (78개 수집됨)
📄 6 페이지 크롤링 중... (98개 수집됨)
📄 7 페이지 크롤링 중... (116개 수집됨)
📄 8 페이지 크롤링 중... (136개 수집됨)
📄 9 페이지 크롤링 중... (154개 수집됨)
📄 10 페이지 크롤링 중... (174개 수집됨)
📄 11 페이지 크롤링 중... (194개 수집됨)
📄 12 페이지 크롤링 중... (214개 수집됨)
📄 13 페이지 크롤링 중... (234개 수집됨)
📄 14 페이지 크롤링 중... (253개 수집됨)
📄 15 페이지 크롤링 중... (273개 수집됨)
📄 16 페이지 크롤링 중... (292개 수집됨)
📄 17 페이지 크롤링 중... (312개 수집됨)
📄 18 페이지 크롤링 중... (331개 수집됨)
📄 19 페이지 크롤링 중... (351개 수집됨)
📄 20 페이지 크롤링 중... (369개 수집됨)
📄 21 페이지 크롤링 중... (389개 수집됨)
📄 22 페이지 크롤링 중... (405개 수집됨)
📄 23 페이지 크롤링 중... (425개 수집됨)
📄 24 페이지 크롤링 중... (445개 수집됨)
📄 25 페이지 크롤링 중... (465개 수집됨)
📄 26 페이지 크롤링 중... (485개 수집됨)
📄 27 페이지 크롤링 중... (505개 수집됨)
📄 28 페이지 크롤링 중... (525개 수집됨)
📄 29 페이지 크롤링 중... (545개 수집됨)
📄 30 페이지 크롤링 중... (565개 수집됨)
📄 31 페이지 크롤링 중... (584개 수집됨)
📄 32 페이지 크롤링 중... (604개 수집됨)
📄 33 페이지 크롤링 중... (624개 수집됨)
📄 34 페이지 크롤링 중... (643개 수집됨)
📄 35 페이지 크롤링 중... (661개 수집됨)
📄

저장하시겠습니까? (y/n):  y


✅ 저장 완료: news_dataset_economy.csv


In [5]:
import pandas as pd

df = pd.read_csv('news_dataset_economy.csv')
df

,title,content,date,press,url
0,"볼빅, 장석헌 신임 대표이사 선임",골프 브랜드 볼빅이 장석헌 신임 대표이사를 선임했다고 26일 밝혔다.\n장석헌 대표...,2026-03-26 22:22:00,조선비즈,https://n.news.naver.com/mnews/article/366/000...
1,"정유업계, 2차 최고가격제 협조…""석유제품 국내 우선 공급""","정부, 2차 석유 최고가격제 27일 0시부터 시행\n(서울=뉴스1) 원태성 기자 =...",2026-03-26 22:18:00,뉴스1,https://n.news.naver.com/mnews/article/421/000...
2,한-브라질 외교장관회담…조현 '對韓 원유수출 확대' 협력제안,(서울=연합뉴스) 민선희 기자 = 조현 외교부 장관은 26일(현지시간) 프랑스에서 ...,2026-03-26 22:17:00,연합뉴스,https://n.news.naver.com/mnews/article/001/001...
3,"목돈부담 큰 계약금 마련 지원…코인베이스, 코인담보 모기지 출시","코인베이스, ‘페니메이 승인 모기지‘ 베터홈앤드파이낸스와 협업\n비트코인·USDC ...",2026-03-26 22:16:00,이데일리,https://n.news.naver.com/mnews/article/018/000...
4,정유업계 “2차 최고가격제 적극 협조…가격·수급 안정 총력”,대한석유협회와 국내 정유업계는 26일 정부가 발표한 비상 경제 대응 방안에 따라 가...,2026-03-26 22:16:00,조선비즈,https://n.news.naver.com/mnews/article/366/000...
...,...,...,...,...,...
995,KB금융 회추위 다음달 가동…양종희 회장 연임 여부 촉각,KB금융지주가 양종희 KB금융 회장의 첫 번째 임기 만료를 앞두고 사외이사진 구성을...,2026-03-26 17:41:00,매일경제,https://n.news.naver.com/mnews/article/009/000...
996,"와디즈, 신규 메이커 수수료 30% 할인…AI·데이터 기능도 강화",첫 펀딩 수수료 30% 할인·광고비 지원\n[데일리안 = 황지현 기자] 와디즈가 신...,2026-03-26 17:41:00,데일리안,https://n.news.naver.com/mnews/article/119/000...
997,"디엔에프, 임정훈 대표이사로 변경",[이데일리 박순엽 기자] 디엔에프(092070)는 기존 신재호 대표이사가 사임함에 ...,2026-03-26 17:41:00,이데일리,https://n.news.naver.com/mnews/article/018/000...
998,'참호구축' 논란에도…진옥동·빈대인 회장 연임 확정,신한·BNK금융 정기 주총 … 진 88%·빈 91% 찬성\n李대통령 '이너서클' 지...,2026-03-26 17:41:00,매일경제,https://n.news.naver.com/mnews/article/009/000...


In [10]:
df['url'].iloc[0]

'https://n.news.naver.com/mnews/article/366/0001152000?sid=101'

In [11]:
df['content'].iloc[0]

'골프 브랜드 볼빅이 장석헌 신임 대표이사를 선임했다고 26일 밝혔다.\n장석헌 대표이사는 서울대 국제경제학과 출신으로 장기신용은행, 한국기술투자, 코아매직, 현대렌탈서비스 등에서 최고재무책임자(CFO)를 역임했다. 2023년 4월부터 볼빅 경영총괄을 맡아 회사 운영을 이끌어왔다.\n장 대표는 “볼빅이 국내에서 쌓아온 위상을 바탕으로 글로벌 시장에서도 경쟁력을 강화해 나갈 것”이라며 “건전한 경영 문화를 정착시키고, 신뢰와 혁신을 기반으로 지속 가능한 성장을 이루겠다”고 말했다.'